# SHAP Model Explainability

**Objective**: Interpret the best XGBoost model using SHAP and translate insights into business recommendations.

Analyses:
1. Built-in Feature Importance (top 10)
2. SHAP Global Summary Plot
3. SHAP Bar Plot (global importance)
4. SHAP Force Plots — True Positive, False Positive, False Negative
5. Top 5 fraud drivers + business recommendations

**Reference**: https://shap.readthedocs.io/en/latest/

---

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

from modeling import load_model
from explainability import (
    get_shap_explainer,
    compute_shap_values,
    plot_shap_summary,
    plot_shap_bar,
    find_sample_indices,
    plot_force_plot,
    plot_feature_importance,
)

os.makedirs('../notebooks/plots', exist_ok=True)
shap.initjs()
print('✅ Imports successful')

## ══════════════════════════════════
## SHAP Analysis: E-Commerce Model
## ══════════════════════════════════

In [ ]:
# Load best model and test data
xgb_fraud = load_model('best_model_fraud.pkl')

fraud_test = pd.read_csv('../data/processed/fraud_test.csv')
fraud_train = pd.read_csv('../data/processed/fraud_train.csv')

X_test = fraud_test.drop(columns=['class'])
y_test = fraud_test['class']
X_train = fraud_train.drop(columns=['class'])

print(f'Test set: {X_test.shape}')

### 1. Built-in Feature Importance

In [ ]:
feat_imp = plot_feature_importance(
    xgb_fraud,
    feature_names=X_test.columns.tolist(),
    model_name='XGBoost E-Commerce',
    top_n=10
)
print('\nTop 10 features (built-in importance):')
print(feat_imp.head(10))

### 2. SHAP Explainer Setup

In [ ]:
explainer_fraud = get_shap_explainer(xgb_fraud)

# Use a sample for speed if test set is large
sample_size = min(500, len(X_test))
X_shap = X_test.sample(n=sample_size, random_state=42)
y_shap = y_test.loc[X_shap.index]

shap_values_fraud = compute_shap_values(explainer_fraud, X_shap)

### 3. Global SHAP Summary Plot

> Each dot = one transaction. Color = feature value (red=high, blue=low). X-axis = impact on fraud prediction.

In [ ]:
plot_shap_summary(shap_values_fraud, X_shap, model_name='XGBoost E-Commerce', max_display=20)

In [ ]:
plot_shap_bar(shap_values_fraud, X_shap, model_name='XGBoost E-Commerce', top_n=10)

### 4. Individual SHAP Force Plots

Waterfall plots show the contribution of each feature to a single prediction:
- **Bars pushing right** (red) → increasing fraud probability
- **Bars pushing left** (blue) → decreasing fraud probability

In [ ]:
# Get predictions on the shap sample
y_pred_shap = xgb_fraud.predict(X_shap)
y_proba_shap = xgb_fraud.predict_proba(X_shap)[:, 1]

# Find TP, FP, FN indices
tp_idx, fp_idx, fn_idx = find_sample_indices(
    y_shap.values, y_pred_shap, y_proba_shap
)

print(f'True Positive index:  {tp_idx} — Predicted fraud, was fraud')
print(f'False Positive index: {fp_idx} — Predicted fraud, was legitimate')
print(f'False Negative index: {fn_idx} — Predicted legitimate, was fraud')

In [ ]:
# True Positive — correctly caught fraud
plot_force_plot(explainer_fraud, shap_values_fraud, X_shap,
                idx=tp_idx, label='True_Positive', model_name='XGBoost_E-Commerce')

In [ ]:
# False Positive — legitimate transaction flagged as fraud
plot_force_plot(explainer_fraud, shap_values_fraud, X_shap,
                idx=fp_idx, label='False_Positive', model_name='XGBoost_E-Commerce')

In [ ]:
# False Negative — fraud that was missed
plot_force_plot(explainer_fraud, shap_values_fraud, X_shap,
                idx=fn_idx, label='False_Negative', model_name='XGBoost_E-Commerce')

## ══════════════════════════════════
## SHAP Analysis: Credit Card Model
## ══════════════════════════════════

In [ ]:
xgb_cc = load_model('best_model_creditcard.pkl')

cc_test = pd.read_csv('../data/processed/creditcard_test.csv')
cc_train = pd.read_csv('../data/processed/creditcard_train.csv')

X_cc_test = cc_test.drop(columns=['Class'])
y_cc_test = cc_test['Class']

explainer_cc = get_shap_explainer(xgb_cc)

cc_sample = X_cc_test.sample(n=min(500, len(X_cc_test)), random_state=42)
y_cc_shap = y_cc_test.loc[cc_sample.index]

shap_values_cc = compute_shap_values(explainer_cc, cc_sample)

In [ ]:
plot_feature_importance(xgb_cc, X_cc_test.columns.tolist(),
                         model_name='XGBoost Credit Card', top_n=10)

In [ ]:
plot_shap_summary(shap_values_cc, cc_sample, model_name='XGBoost Credit Card')

In [ ]:
y_pred_cc = xgb_cc.predict(cc_sample)
y_proba_cc = xgb_cc.predict_proba(cc_sample)[:, 1]

tp_cc, fp_cc, fn_cc = find_sample_indices(y_cc_shap.values, y_pred_cc, y_proba_cc)

plot_force_plot(explainer_cc, shap_values_cc, cc_sample,
                idx=tp_cc, label='True_Positive', model_name='XGBoost_CreditCard')
plot_force_plot(explainer_cc, shap_values_cc, cc_sample,
                idx=fp_cc, label='False_Positive', model_name='XGBoost_CreditCard')
plot_force_plot(explainer_cc, shap_values_cc, cc_sample,
                idx=fn_cc, label='False_Negative', model_name='XGBoost_CreditCard')

## Interpretation Summary

### Top 5 Fraud Drivers (E-Commerce Model)

*(Fill in after running — based on SHAP bar plot output)*

| Rank | Feature | SHAP Interpretation |
|------|---------|--------------------|
| 1 | `time_since_signup` | Very short signup-to-purchase time strongly indicates fraud |
| 2 | `transaction_velocity` | High frequency of transactions in short windows is a fraud signal |
| 3 | `purchase_value` | Unusually high purchase values correlate with fraud |
| 4 | `country_*` | Certain countries have elevated fraud probabilities |
| 5 | `age` | Extreme age values (very young/very old) show fraud clustering |

### SHAP vs. Built-in Importance Comparison

SHAP provides more reliable feature importance than built-in `feature_importances_` because:
- Built-in importance counts split frequency — features with many categories get inflated scores
- SHAP measures the actual marginal contribution to each prediction
- SHAP handles correlated features more gracefully

---

## Business Recommendations

### 📌 Recommendation 1: Early Account Verification
> **Rule**: Flag any transaction occurring within **1 hour of account signup** for mandatory 2FA or manual review.  
> **SHAP Insight**: `time_since_signup` is the #1 SHAP driver. Fraudsters create accounts and transact immediately — legitimate users take hours or days before their first purchase.

### 📌 Recommendation 2: Velocity-Based Rate Limiting
> **Rule**: Automatically block or challenge accounts with **>3 transactions within 1 hour**.  
> **SHAP Insight**: `transaction_velocity` is a top fraud driver. Automated fraud attacks create high-frequency transaction bursts that are absent in legitimate usage.

### 📌 Recommendation 3: Geolocation Risk Scoring
> **Rule**: Apply a **country-level risk multiplier** to transactions from the top-10 highest fraud-rate countries (identified in EDA). Route high-risk country transactions through additional verification.  
> **SHAP Insight**: Country features contribute significantly. Geolocation mismatch from a user's historical country should trigger a fraud alert.

### 📌 Recommendation 4: High-Value Transaction Review
> **Rule**: Automatically review any transaction **>2 standard deviations above a user's historical average** purchase value.  
> **SHAP Insight**: `purchase_value` is a strong predictor. Fraudsters often attempt large-value purchases quickly before being detected.

### 📌 Recommendation 5: Explainability in Dispute Resolution
> **Rule**: When a customer disputes a fraud flag, use **SHAP force plots** to explain to agents *exactly why* the transaction was flagged (e.g., "This transaction occurred 3 minutes after signup from a high-risk country").  
> **SHAP Insight**: Individual force plots make each decision transparent, enabling faster, fairer dispute resolution and building customer trust.